# xGEMS tutorial 2 — titrating basalt into seawater (250 °C, 1000 bar)

**Authors: G. Dan Miron**

**Formatted with Claude Opus**

Into **1 kg of seawater** we add 1 → 500 g of basalt glass, re-equilibrating each time, and
plot the **volume fraction of each solid phase**.

The whole titration is one idea — rescale basalt and add it to seawater:

```python
mix = seawater + basalt.set_quantity(n, "g")
gems.equilibrate(T, P, mix)
```

The y-axis is each solid's share of the **total solids volume**

```
cV = V(system) − V(aq_gen) − V(fluid_gen)     # everything that is not fluid
yp = V(phase) / cV
```

> Runs against your local `xgems` env + `gems_files/SW-B_titr3-dat.lst`; not executed here.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgems import ChemicalEngineDicts, Material

gems = ChemicalEngineDicts("gems_files/SW-B_titr3-dat.lst")

T = 250.0 + 273.15     # K   (250 °C)
P = 1000e5             # Pa  (1000 bar)

## 2. Seawater and basalt (as in tutorial 1)

In [ ]:
basalt = Material(gems, "basalt_glass")
basalt.add("SiO2",  49.6151, "g")   # Silica
basalt.add("Al2O3", 14.7342, "g")   # Alumina
basalt.add("CaO",   12.5028, "g")   # Lime
basalt.add("FeO",   10.0261, "g")   # Ferrous oxide
basalt.add("MgO",    8.6533, "g")   # Magnesia
basalt.add("Na2O",   2.0472, "g")   # Soda
basalt.add("TiO2",    1.3190, "g")  # Titania
basalt.add("Fe2O3",   0.7912, "g")  # Ferric oxide
basalt.add("K2O",     0.3111, "g")  # Potash

metals = Material(gems, "metals")
metals.add({
    "Ag": 7.655346e-07,
    "Au": 4.192420e-07,
    "Cu": 1.299481e-03,
    "Pb": 3.985369e-03,
    "Zn": 1.263029e-02
})

basalt_metals = basalt + metals

water = Material(gems, "water")
water.add("H2O", 1000.0, "g")       # 55.5093 mol H2O

salts = Material(gems, "sea_salts")
salts.add({
    "C":  0.002037888,
    "Ca": 0.01075073,
    "Cl": 0.5666381,
    "K":  0.01057964,
    "Mg": 0.05474226,
    "Na": 0.4860599,
    "S":  0.02926366,
    "O":  0.1223337,
})

seawater = water + salts

## 3. Titration — 1 → 500 g of basalt

The solid phases we track (your GEMS phase list). At each step we read every phase volume,
work out the total solids volume `cV`, and store each solid's fraction `V(phase) / cV`.

In [ ]:
SOLID_PHASES = [
    "Alkali Feldspar", "Plagioclase", "Chlorite", "Cpx", "Epidote", "Quartz",
    "Titanite", "Hematite", "Magnetite", "Prehnite", "Anhydrite",
    "Gold", "Silver", "Acanthite", "Bornite", "Chalcocite", "Chalcopyrite",
    "Covellite", "Galena", "Pyrite", "Sphalerite", "Dolomite", "Calcite", "Paragonite"
]

grams = range(50, 501,10)          # 1 to 500 g of basalt


saponites = ["Ca-Saponite", "Fe-Saponite", "K-Saponite",
             "Mg-Saponite", "Na-Saponite"]

gems.suppress_multiple_phases(saponites, min_amount=0.0, max_amount=0.0)

rows = []
for n in grams:
    mix = seawater.set_quantity(1.0, "kg") + basalt_metals.set_quantity(n, "g")
    gems.cold_start()
    gems.equilibrate(T, P, mix)

    pv = gems.phases_volume                                   # {phase: volume in m3}
    cV = gems.system_volume - pv["aq_gen"] - pv["fluid_gen"]  # total solids volume
    rows.append({p: pv.get(p, 0.0) / cV for p in SOLID_PHASES})

yp = pd.DataFrame(rows, index=list(grams))     # volume fraction of solids
yp.index.name = "g_basalt"
yp.head()

## 4. Plot — solid volume fractions vs grams of basalt

Stacked areas, styled like the GEMS titration plot. Phases that never appear (e.g. the trace ore minerals) are left out of the legend.

In [ ]:
present = [p for p in SOLID_PHASES if yp[p].max() > 1e-6]   # drop phases that stay at 0

ax = yp[present].plot.area(figsize=(8, 6), cmap="tab20", linewidth=0)
ax.set_xlim(50, 500)          # match the GEMS figure (drop the noisy low-basalt end)
ax.set_ylim(0, 1.10)
ax.set_xlabel("Basalt (g)")
ax.set_ylabel("Vol (%)")      # fraction of total solids (0–1), as in GEM-Selektor
ax.set_title("Seawater–basalt titration model 250 °C / 1 kbar")
ax.legend(fontsize=8, ncol=2, loc="lower right")
plt.tight_layout(); plt.show()

## 5. Same result vs fluid/rock ratio

Water–rock results are usually shown against the mass **fluid/rock ratio**. Fluid = 1 kg
seawater, rock = grams of basalt, so F/R = 1000 / (g basalt); fluid-dominated is on the left.

In [ ]:
fr = 1000.0 / np.array(list(grams))     # kg seawater per kg basalt

yp_fr = yp[present].copy()
yp_fr.index = fr

ax = yp_fr.plot.area(figsize=(8, 6), cmap="tab20", linewidth=0)
ax.set_xscale("log")
ax.set_xlim( fr.min(), 10)   # 50 g -> 500 g window, fluid-dominated left
ax.set_ylim(0, 1.30)
ax.set_xlabel("fluid / rock  (kg seawater per kg basalt)")
ax.set_ylabel("Vol (%)")
ax.set_title("Solid assemblage vs fluid/rock ratio")
ax.legend(fontsize=8, ncol=2, loc="upper left")
plt.tight_layout(); plt.show()

## 6. Same titration, now with CO₂

Add a fixed amount of CO₂ to the mix at every step — one extra `Material` in the sum.

In [ ]:
co2 = Material(gems, "co2")
co2.add("CO2", 10.0, "g")        # 10 g CO2 per kg seawater

rows = []
for n in grams:
    mix = seawater + basalt_metals.set_quantity(n, "g") + co2
    gems.equilibrate(T, P, mix)
    pv = gems.phases_volume
    cV = gems.system_volume - pv["aq_gen"] - pv["fluid_gen"]
    rows.append({p: pv.get(p, 0.0) / cV for p in SOLID_PHASES})

yp_co2 = pd.DataFrame(rows, index=list(grams))
present_co2 = [p for p in SOLID_PHASES if yp_co2[p].max() > 1e-6]

ax = yp_co2[present_co2].plot.area(figsize=(8, 6), cmap="tab20", linewidth=0)
ax.set_xlim(50, 500); ax.set_ylim(0, 1.40)
ax.set_xlabel("Basalt (g)"); ax.set_ylabel("Vol (%)")
ax.set_title("With 10 g CO₂ per kg seawater — 250 °C / 1 kbar")
ax.legend(fontsize=8, ncol=2, loc="upper left")
plt.tight_layout(); plt.show()